# TP 5 : Tests de Primalité (partie 2 : test de Miller–Rabin)

Dans la première partie, on a vu que le test de Fermat échoue sur les nombres de Carmichael : pour ces entiers, toutes les bases premières avec $N$ sont des menteurs, et augmenter le nombre $k$ de bases testées ne suffit pas.

Le test de Miller–Rabin corrige ce défaut en exploitant une propriété supplémentaire des nombres premiers : modulo un nombre premier, $1$ n'a que deux racines carrées, $1$ et $-1$. L'objectif de ce TP est d'étudier cette propriété, d'implémenter le test de Miller–Rabin et de quantifier sa probabilité d'erreur.

## Exercice 1 : Polynômes dans $\mathbb Z/p\mathbb Z$ (Introduction)

**Question 1** Ecrire une fonction ``cherche_racine_carree(n,a)`` qui prend en argument des entiers $a$ et $n$ et qui renvoie la liste des entiers $0 \leq x \leq n-1$ tels que $x^2 \equiv a \mod n$.


On a vu en cours que pour tout entier $n \in \mathbb Z$, l'ensemble $\mathbb Z/n\mathbb Z$ est un anneau. On définit comme sur un corps les _polynômes à valeur dans_ $\mathbb Z/n\mathbb Z$, et on note $(\mathbb Z/n\mathbb Z)[X]$  l'ensemble de ces polynômes. 

Dans ce contexte, chercher les nombre $x$ qui vérifient $x^2 \equiv a \mod n$ revient à trouver les racines du polynôme $X^2 - a$ dans $\mathbb Z/n\mathbb Z$.

**Question 2.** Pour $n$ allant de $2$ à $1000$, calculer le nombre de racines du polynôme $X^2 - 1$ dans $\mathbb Z/n\mathbb Z$. En quoi est-ce étonnant ?

On a vu en cours que lorsque $n$ est premier alors $\mathbb Z/n\mathbb Z$ est un corps. La théorie des polynômes sur des corps est très riche, et nous fournit une propriété très intéressante : si $P$ est un polynôme de degré $d$ à coefficients dans un corps $k$, alors $P$ admet au plus $d$ racines sur $k$.

**Question 3.** Soit $p$ un nombre premier. Quelles sont les racines de $X^2-1$ dans $\mathbb Z/p\mathbb Z$ ? _On pourra utiliser le programme de la question 1. pour conjecturer une forme générale, avant de la démontrer._

À l'inverse, cette propriété n'est pas toujours vérifiée pour un anneau quelconque. Par exemple, dans $\mathbb Z/15\mathbb Z$, on a $1^2 \equiv 14^2 \equiv 4^2 \equiv 11^2\equiv 1 \mod 15$. Plus généralement, on peut montrer que si $n$ est un nombre composé impair alors le polynôme $X^2-1$ aura au moins $3$ racines dans $\mathbb Z/n\mathbb Z$.

**Question 4.** En déduire un programme ``est_premier_pol(n)`` qui détermine si un entier $n$ est premier en utilisant le principe ci-dessus. Quelle est la complexité de votre algorithme ? 

_Pour aller plus loin : La propriété fondamentale utilisée ici est l'_ intégrité _de $\mathbb Z/p\mathbb Z$._

_Pour tout corps $k$, l'anneau $k[X]$ est un anneau_ euclidien _(voir TP3), ce qui permet de définir une notion de division euclidienne dessus. Grâce à cette division euclidienne, on gagne un certain nombre de bonnes propriétés sur notre anneau dont la principalité (tout idéal $I \subset k[X]$ est de la forme $I = P k[X]$ pour un élément $P \in k[X]$) et la factorialité (tout polynôme s'écrit de manière unique comme produit de polynômes irréductibles). Avec ces deux propriétés, on déduit que tout polynôme de degré $d$ admet au plus $d$ racines sur $k$._

_Plus généralement, si $A$ est un anneau intègre, alors on peut définir $k$ son corps des fractions. Comme $k$ est un corps, alors tout polynôme de degré $d$ à coefficients dans $k$ admet au plus $d$ racines sur $k$. Mais comme $A \subset k$ par construction, on en déduit que cette propriété reste vraie dans $A$._

_À l'inverse, si $A$ n'est pas intègre, on peut fixer deux éléments $a,b \in A$ non nuls tels que $ab = 0$. On définit alors $P = (X-a)(X-b) \in A[X]$. Par construction, on a $P(a) = P(b) =0$, et de plus $P(0) = (0-a)(0-b) = ab = 0$. Ainsi $P$ est un polynôme de degré $2$ qui admet $3$ racines dans $A$. Dans $\mathbb Z/15\mathbb Z$, par exemple, le polynôme $3X$ a pour racines $0$, $5$ et $10$._

## Exercice 2 - L'algorithme de Miller-Rabin

Soit $N$ un entier. On cherche à rafiner le test de Fermat en utilisant le principe introduit dans l'exercice précédent. 

Dans tout cet exercice on va considérer que $N$ est impair (s'il est pair, on verifie immédiatement s'il est premier ou non), et on écrira $N-1 = 2^v\cdot m$ avec $m$ un entier impair. Notez que $v \geqslant 1$ car $N-1$ est pair.



Soit $2 \leqslant a \leqslant N-1$. L'idée du test de Miller-Rabin est la suivante :
* calculer la suite $(b_i)_{0\leqslant i \leqslant v}$ définie par :
$$ b_0 = a^m \mod N \, \text{ et }\, b_{i+1} = b_i^2 \mod N$$

En particulier on a $b_v = a^{m \cdot 2^v} = a^{N-1}$.

**Si $N$ est premier**, le petit théorème de Fermat donne $b_v = 1$. Comme $b_{v-1}^2 \equiv b_v = 1 \pmod N$, l'exercice 1 montre que $b_{v-1} = 1$ ou $b_{v-1} = N - 1$. Si $b_{v-1} = 1$, le même raisonnement s'applique à $b_{v-2}$, et ainsi de suite. La suite est donc de l'une des deux formes suivantes :
* $b_0 = b_1 = \dots = b_v = 1$ ;
* il existe $i \le v - 1$ tel que $b_i = N - 1$ (et alors $b_{i+1} = \dots = b_v = 1$).

Comment ce test fonctionne-t-il ? 


**Si la suite n'a aucune de ces deux formes**, $N$ est donc certainement composé. Deux cas peuvent se produire :
* $b_v \neq 1$ : le petit théorème de Fermat est mis en défaut, comme dans le test de Fermat ;
* il existe $i \ge 1$ tel que $b_i = 1$ et $b_{i-1} \notin \{1, N-1\}$ : alors $b_{i-1}$ est une racine carrée de $1$ différente de $\pm 1$, ce qui est impossible modulo un nombre premier d'après l'exercice 1.

**Si la suite a l'une des deux formes**, on ne peut pas conclure : $N$ est *probablement premier*.

Le test de Fermat ne regarde que $b_v$ ; le test de Miller–Rabin examine toute la suite. Il détecte donc tous les composés que détecte le test de Fermat, et d'autres en plus.


### Exercice 2.a
**Question 1.** Écrire une fonction `decompose(N)` qui renvoie deux entiers $v$ et $m$, avec $m$ impair, tels que $N-1 = 2^v⋅m$.

**Question 2.** Écrire une fonction `SuiteB(N,a)` prend en entrée un entier $N$ impair et un entier $a$ entre $2$ et $N-1$, et renvoie la liste $[b_0, …, b_v]$ comme définie ci-dessus.

**Question 3.** Écrire une fonction `testSuiteB(N, a)` qui calcule la suite de $b_i$ associée à $a$ en appelant `SuiteB(N)`, et qui renvoie `False` si $N$ n'est certainement pas premier selon les critères donnés ci-dessus, et `True` sinon. 

**Question 4.** Testez votre fonction pour quelques nombres premiers $p$ et toutes les bases $2 \leqslant a \leqslant p-1$. 

### Exercice 2.b — Menteurs forts

Pour $N$ composé, on appelle *menteur fort* pour $N$ un entier $a \in [1, N-1]$ premier avec $N$ pour lequel la suite $(b_i)$ a l'une des deux formes du cas premier. Tout menteur fort est un menteur de Fermat, puisque $b_v = 1$ dans les deux cas.

**Question 1.** Écrire une fonction `menteurs_forts(N)` qui renvoie la liste des menteurs forts pour $N$.


**Question 2.** Pour les nombres de Carmichael $561$, $1105$, $1729$, $2465$ et $2821$, puis pour quelques entiers composés $N \in [1000, 5000]$, comparer la proportion de menteurs forts $\#\{\text{menteurs forts}\} / \varphi(N)$ à la proportion de menteurs de Fermat. Qu'observez-vous ?


### Exercice 2.c
L'algorithme de Miller-Rabin est quasiment écrit : il s'agit de tirer aléatoirement plusieurs valeurs $a$ aléatoirement, de calculer à chaque fois la suite $(b_i)$ et de vérifier si elle vérifie les conditions des nombres premiers. Si toutes les suites $(b_i)$ vérifient les conditions, l'algorithme répond `True` (le nombre est premier) sinon il répond `False`.

**Question 1.** Implanter l'algorithme de Miller-Rabin avec une fonction `Test_Miller_Rabin(N,k)` qui choisit aléatoirement $k$ entiers $a$ pour appeler la fonction `testSuiteB`, et retourne `False` si $N$ est certainement composé et `True` s'il est probablement premier.

**Question 2.** Estimez le temps de calcul de votre algorithme en fonction de $k$ et de $M(N)$ (le temps de calcul pour une multiplication de deux nombres $<N$) et $D(N)$ (le temps de calcul pour une réduction modulo $N$).

Essayez d'optimiser votre programme (quitte à ne pas utiliser la fonction `testSuiteB`) pour effectuer le moins possible d'opérations. _En pratique, on effectue ce test sur des nombre dont la taille peut aller jusqu'à plusieurs milliers de bits, donc $M(N)$ et $D(N)$ sont très importants, et on essaie de minimiser les constantes devant._

**Question 3.** Tester la fonction sur différents nombres (premiers, composés, Carmichael), avec différentes valeurs de $k$ (entre $1$ et $10$ par exemple). Essayer de répondre aux questions suivantes :
* Arrive-t-il à la fonction de se tromper quand $N$ est premier ? et quand $N$ est composé ? et quand il est de type Carmichaël ?
* Dans les cas où il arrive que la fonction se trompe, avec quelle probabilité (empirique) cela arrive-t-il ?